# 03 — Huấn luyện và chạy pipeline hoàn chỉnh

Notebook gốc bước 0–8 được **chuyển thành các hàm/module trong `src/`**. Notebook này gọi `run_pipeline.main()` thay vì chép toàn bộ hàng trăm dòng lần nữa. Bằng cách đó notebook và lệnh `python run_pipeline.py` chạy đúng **cùng một mã nguồn**: đọc dataset → tiền xử lý → Isolation Forest + Autoencoder → hiệu chuẩn ngưỡng → test và baseline (nếu bật) → xuất artifacts.

**Lưu ý:** chạy hết notebook này sẽ tốn RAM/thời gian, và ghi đè outputs cũ.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


Project: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode
Dataset: D:\network-anomaly-hybrid-vscode\network-anomaly-hybrid-vscode\data\raw


In [2]:
print('SAMPLE:', cfg.SAMPLE, '| TEST_FRAC:', cfg.TEST_FRAC,
      '| WITH_SUPERVISED:', cfg.WITH_SUPERVISED)
print('Mục tiêu FPR:', cfg.FPR_TARGET)
print('Để đổi cấu hình, đặt biến môi trường trước khi mở kernel, rồi Restart Kernel.')


SAMPLE: 300000 | TEST_FRAC: 0.25 | WITH_SUPERVISED: True
Mục tiêu FPR: 0.01
Để đổi cấu hình, đặt biến môi trường trước khi mở kernel, rồi Restart Kernel.


In [3]:
from run_pipeline import main
main()


Train: ['2018-02-14', '2018-02-15'] Validation: ['2018-02-16'] Test: ['2018-02-20', '2018-02-21', '2018-02-22', '2018-02-23', '2018-02-28', '2018-03-01', '2018-03-02']
[21:12:13] Ingest and prepare parquet
[21:13:15] Benign training 1,268,743 -> 1,268,738 records
[21:13:27] Training models on 300,035 benign records
[21:16:04] Đã huấn luyện IF + AE trên mẫu 300,035 x 46; AE (46, 46, 23, 11, 23, 46, 46)
[21:16:04] Models trained in 156.5s
[21:16:17] Calibrated threshold: {'s_iso': 0.5815025935353016, 's_ae': 2.5954462337493904, 's_ens': 5.3479444711721795}
           model         day       n  attack_rate  recall    fpr  precision  \
0  GBT cross-day  2018-02-20  239192        0.603   0.000  0.001      0.000   
1  GBT cross-day  2018-02-21  139945        0.358   0.000  0.000      0.000   
2  GBT cross-day  2018-02-22  208226        0.000   0.000  0.001      0.000   
3  GBT cross-day  2018-02-23  207464        0.001   0.000  0.001      0.000   
4  GBT cross-day  2018-02-28  114426        

In [4]:
import json
import pandas as pd
from IPython.display import display
p = cfg.OUT_DIR / 'metrics' / 'metrics_by_day.csv'
if p.exists():
    display(pd.read_csv(p).round(4))
print('Models:', list((cfg.OUT_DIR / 'models').glob('*.joblib')))
print('Figures:', list((cfg.OUT_DIR / 'figures').glob('*.png')))


,model,day,n,attack_rate,recall,fpr,precision,f1,roc_auc,pr_auc
0,Isolation Forest,2018-02-20,239192,0.6032,0.0000,0.0307,0.0021,0.0001,0.7488,0.6992
1,Isolation Forest,2018-02-21,139945,0.3577,0.0001,0.0052,0.0128,0.0002,0.0587,0.2103
2,Isolation Forest,2018-02-22,208226,0.0004,0.1071,0.0263,0.0016,0.0032,0.7596,0.0012
3,Isolation Forest,2018-02-23,207464,0.0007,0.3475,0.0265,0.0088,0.0172,0.8293,0.0070
4,Isolation Forest,2018-02-28,114426,0.1238,0.0523,0.0559,0.1167,0.0722,0.5070,0.1244
5,Isolation Forest,2018-03-01,62659,0.2476,0.0743,0.0527,0.3171,0.1204,0.5155,0.2681
6,Isolation Forest,2018-03-02,192876,0.1870,0.0067,0.0340,0.0436,0.0117,0.8795,0.4392
7,Isolation Forest,ALL,1164788,0.2235,0.0085,0.0307,0.0736,0.0152,0.6697,0.2803
8,Autoencoder,2018-02-20,239192,0.6032,0.0013,0.0000,0.9948,0.0027,0.9633,0.9452
9,Autoencoder,2018-02-21,139945,0.3577,0.0085,0.0000,1.0000,0.0168,0.0088,0.2136


Models: [WindowsPath('D:/network-anomaly-hybrid-vscode/network-anomaly-hybrid-vscode/outputs/models/feature_spec.joblib'), WindowsPath('D:/network-anomaly-hybrid-vscode/network-anomaly-hybrid-vscode/outputs/models/model.joblib')]
Figures: [WindowsPath('D:/network-anomaly-hybrid-vscode/network-anomaly-hybrid-vscode/outputs/figures/ae_loss.png'), WindowsPath('D:/network-anomaly-hybrid-vscode/network-anomaly-hybrid-vscode/outputs/figures/metrics_by_day.png')]
